# Praktikum 6 - Pencarian Lagu Mirip (Spotify)

**Nama:** Radit Firansah  
**NIM:** 244107020196

Membandingkan ANNOY, FAISS, dan HNSW pada dataset Spotify (`spotify_songs.csv`).

In [1]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/spotify_songs.csv')
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10

print("Jumlah track:", X_scaled.shape[0])
print("Jumlah fitur:", X_scaled.shape[1])

Jumlah track: 32833
Jumlah fitur: 9


### Exact Nearest Neighbor (brute-force)

In [2]:
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

Exact NN done in 2.292 s


### Annoy

In [3]:
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

Annoy done in 0.624 s


### HNSW

In [4]:
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

HNSW done in 0.446 s


### FAISS IVF

In [5]:
start = time.time()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
dist_faiss, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

FAISS IVF done in 0.227 s


### Hasil

In [6]:
print("Top-5 neighbors for first song:")
print(f"Exact NN: {idx_exact[0][:5]}")
print(f"Annoy:    {idx_annoy[0][:5]}")
print(f"HNSW:     {idx_hnsw[0][:5]}")
print(f"FAISS:    {idx_faiss[0][:5]}")

print("\nJudul lagu pertama:", df['track_name'].iloc[0])
print("\nTetangga terdekat (Exact NN):")
for i in idx_exact[0][:5]:
    print(f"  {df['track_name'].iloc[i]} - {df['track_artist'].iloc[i]}")

Top-5 neighbors for first song:
Exact NN: [29684     0 29734  4959 29713]
Annoy:    [0, 29684, 29734, 4959, 29713]
HNSW:     [    0 29684 29734  4959 29713]
FAISS:    [    0 29684 29734  4959 29713]

Judul lagu pertama: I Don't Care (with Justin Bieber) - Loud Luxury Remix

Tetangga terdekat (Exact NN):
  I Don't Care (with Justin Bieber) - Loud Luxury Remix - Ed Sheeran
  I Don't Care (with Justin Bieber) - Loud Luxury Remix - Ed Sheeran
  I Miss You (feat. Julia Michaels) - Cahill Remix - Clean Bandit
  Name & Number - Shift K3Y
  Rise - TV Noise Ibiza Mix - Jonas Blue


## Analisa

- **Exact NN (brute-force)** menghitung jarak query ke **seluruh** track, sehingga hasilnya paling akurat tetapi paling lambat ketika data bertambah besar.
- **Annoy** membangun beberapa *random projection tree*. Pencarian hanya memeriksa leaf yang relevan sehingga jauh lebih cepat, dengan hasil yang sangat mirip exact (approximate).
- **HNSW** menggunakan graph hierarkis sehingga biasanya paling cepat di CPU dengan recall tinggi. Pada dataset kecil ini perbedaannya tidak terlalu terlihat, tetapi pada data besar keunggulannya jelas.
- **FAISS IVF** membagi ruang vektor menjadi cluster. Query hanya mencari pada sebagian cluster (`nprobe`), sehingga lebih cepat dari brute-force. Akurasinya bergantung pada `nlist` dan `nprobe`.
- Pada dataset Spotify ini (32.833 track, 9 fitur numerik yang telah distandardisasi), keempat metode menghasilkan tetangga yang mirip. Perbedaan utama terletak pada **waktu komputasi** dan **skalabilitas**, bukan pada kualitas hasil untuk ukuran data ini.
- Semua metode memakai metrik **Euclidean/L2** setelah fitur dinormalisasi dengan `StandardScaler`, agar skala tiap fitur (misal `tempo` vs `danceability`) seimbang.